# Config and Providers

Explore jury config validation and provider resolution rules.

In [ ]:
import sys
from pathlib import Path

repo_src = Path("..").resolve() / "src"
if repo_src.exists():
    sys.path.insert(0, str(repo_src))

from pydantic import ValidationError
from openjury import JuryConfig, OpenJury

In [ ]:
# Load a valid config from examples
cfg_path = Path("../examples/provider_configs/openai_direct.json").resolve()
config = JuryConfig.from_json_file(str(cfg_path))
print("Jury:", config.name)
print("Jurors:", [j.name for j in config.jurors])
print("Criteria:", [c.name for c in config.criteria])

In [ ]:
# Partial juror override fails validation
bad = {
    "name": "Bad Jury",
    "llm_provider": {
        "provider": "openai_compatible",
        "model_name": "gpt-4o-mini",
        "api_key": "test-key",
    },
    "jurors": [{"name": "Bad Juror", "model_name": "gpt-4o"}],
    "criteria": [{"name": "x", "description": "y"}],
}
try:
    JuryConfig.model_validate(bad)
except ValidationError as e:
    print("Expected error:", e.errors()[0]["msg"])

In [ ]:
# Valid per-juror override (all three fields)
good_override = {
    "name": "Mixed Jury",
    "llm_provider": {
        "provider": "openai_compatible",
        "model_name": "gpt-4o-mini",
        "api_key": "test-openai-key",
    },
    "jurors": [
        {"name": "GPT Juror", "weight": 1.0},
        {
            "name": "Override Juror",
            "model_name": "other-model",
            "provider": "openai_compatible",
            "api_key": "test-other-key",
            "base_url": "https://openrouter.ai/api/v1",
            "weight": 1.0,
        },
    ],
    "criteria": [{"name": "accuracy", "description": "Is it correct?"}],
}
mixed = JuryConfig.model_validate(good_override)
print("Valid override config:", mixed.name)
print("Override juror model:", mixed.jurors[1].model_name)

In [ ]:
# Optional: inspect resolved jury summary (requires valid API key env vars)
import os
if os.environ.get("OPENAI_API_KEY"):
    jury = OpenJury(JuryConfig.from_json_file(str(cfg_path)))
    import json
    print(json.dumps(jury.get_summary(), indent=2))
else:
    print("Set OPENAI_API_KEY to initialize OpenJury with live credentials.")